# Skala PySCF / GPU4PySCF Benchmark Results

This notebook loads and compares JSON produced by `benchmarks/run_pyscf_ao_screening_benchmark.py`. It does not construct molecules, load Skala, or execute benchmark workloads.

Generate result files from a shell before opening the analysis cells:

```bash
python benchmarks/run_pyscf_ao_screening_benchmark.py --label mr
python benchmarks/run_pyscf_ao_screening_benchmark.py \
    --label main \
    --source-root /path/to/main-worktree
```

Add `--smoke` to run only C4H10, or `--preflight-only` to validate the selected checkout and environment without collecting measurements.

## Select Result Files

By default, every compatible molecule-benchmark result in `benchmarks/results` is loaded. Results with other schemas, such as rotation comparisons, are reported and ignored. Replace `SELECTED_RESULT_FILES` with an explicit list when comparing only particular labels or commits.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np

MODES = ("cpu", "cpu_dense", "gpu")
MEASUREMENTS = ("runtime", "memory")
TERMINAL_STATUSES = {"ok", "timeout", "oom", "error", "skipped_after_resource_failure"}
SCIENTIFIC_CONFIG_KEYS = (
    "functional",
    "basis",
    "grid_level",
    "grid_alignment",
    "max_memory_mb",
    "cpu_threads",
    "full_carbon_counts",
    "expected_ao_counts",
)


def find_repository_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "pyproject.toml").is_file() and (
            candidate / "benchmarks"
        ).is_dir():
            return candidate
    raise FileNotFoundError(f"Could not find the Skala repository above {start}")


def is_molecule_benchmark_result(path: Path) -> bool:
    document = json.loads(path.read_text(encoding="utf-8"))
    return isinstance(document.get("molecules"), dict)


REPOSITORY_ROOT = find_repository_root(Path.cwd())
RESULTS_DIR = REPOSITORY_ROOT / "benchmarks" / "results"
CANDIDATE_RESULT_FILES = sorted(RESULTS_DIR.glob("skala-pyscf-ao-screening-*.json"))
SELECTED_RESULT_FILES = [
    path for path in CANDIDATE_RESULT_FILES if is_molecule_benchmark_result(path)
]
IGNORED_RESULT_FILES = [
    path for path in CANDIDATE_RESULT_FILES if path not in SELECTED_RESULT_FILES
]

print(f"Selected {len(SELECTED_RESULT_FILES)} result file(s) from {RESULTS_DIR}")
for result_file in SELECTED_RESULT_FILES:
    print(f"  {result_file.name}")
if IGNORED_RESULT_FILES:
    print("Ignored incompatible result file(s):")
    for result_file in IGNORED_RESULT_FILES:
        print(f"  {result_file.name}")

## Load and Validate

The checks below surface incompatible schemas, scientific settings, hardware, routing implementations, dirty checkouts, unexpected statuses, AO counts, and production-versus-CPU-dense fingerprint differences.

In [ ]:
def validate_result_document(document: dict[str, Any]) -> list[str]:
    errors: list[str] = []
    if document.get("schema_version") != 1:
        errors.append(f"Unsupported schema version: {document.get('schema_version')}")
    for formula, molecule in document.get("molecules", {}).items():
        observed = molecule.get("observed")
        if observed is not None:
            if observed.get("actual_aos") != molecule.get("expected_aos"):
                errors.append(
                    f"{formula}: expected {molecule.get('expected_aos')} AOs, "
                    f"observed {observed.get('actual_aos')}"
                )
            carbon_count = int(molecule["carbon_count"])
            expected_electrons = 8 * carbon_count + 2
            if observed.get("electron_count") != expected_electrons:
                errors.append(
                    f"{formula}: expected {expected_electrons} electrons, "
                    f"observed {observed.get('electron_count')}"
                )
        for mode, mode_record in molecule.get("modes", {}).items():
            for measurement in MEASUREMENTS:
                result = mode_record.get(measurement)
                if result is not None and result.get("status") not in TERMINAL_STATUSES:
                    errors.append(
                        f"{formula} {mode} {measurement}: unknown status {result.get('status')}"
                    )
    return errors


def preferred_fingerprint(mode_record: dict[str, Any]) -> dict[str, float] | None:
    for measurement in MEASUREMENTS:
        result = mode_record.get(measurement, {})
        if result.get("status") == "ok" and "fingerprint" in result:
            return result["fingerprint"]
    return None


def fingerprint_warnings(document: dict[str, Any]) -> list[str]:
    messages: list[str] = []
    for formula, molecule in document["molecules"].items():
        reference = preferred_fingerprint(molecule["modes"]["cpu_dense"])
        if reference is None:
            continue
        for production_mode, rtol, atol in (
            ("cpu", 1e-8, 5e-8),
            ("gpu", 1e-7, 2e-7),
        ):
            production = preferred_fingerprint(molecule["modes"][production_mode])
            if production is None:
                continue
            for key in production:
                if not np.isclose(
                    production[key], reference[key], rtol=rtol, atol=atol
                ):
                    messages.append(
                        f"{formula} {production_mode}/cpu_dense: {key} differs "
                        f"({production[key]:.12g} vs {reference[key]:.12g})"
                    )
    return messages


def comparison_warnings(documents: list[dict[str, Any]]) -> list[str]:
    messages: list[str] = []
    if not documents:
        return ["No result documents were selected"]
    reference = documents[0]
    reference_config = reference["configuration"]
    reference_environment = reference["environment"]
    for document in documents:
        label = document["run_label"]
        if document["source"].get("dirty"):
            messages.append(f"{label}: source checkout is dirty")
        messages.extend(
            f"{label}: {error}" for error in validate_result_document(document)
        )
        messages.extend(
            f"{label}: {warning}" for warning in fingerprint_warnings(document)
        )
    for document in documents[1:]:
        label = document["run_label"]
        for key in SCIENTIFIC_CONFIG_KEYS:
            if document["configuration"].get(key) != reference_config.get(key):
                messages.append(f"{label}: configuration differs for {key}")
        for key_path in (("hostname",), ("platform",), ("cuda", "device_name")):
            left: Any = reference_environment
            right: Any = document["environment"]
            for key in key_path:
                left = left.get(key) if isinstance(left, dict) else None
                right = right.get(key) if isinstance(right, dict) else None
            if left != right:
                messages.append(
                    f"{label}: environment differs for {'.'.join(key_path)}"
                )

    route_implementations: dict[str, set[str]] = {mode: set() for mode in MODES}
    for document in documents:
        for molecule in document["molecules"].values():
            for mode in MODES:
                implementation = (
                    molecule["modes"][mode].get("route", {}).get("implementation")
                )
                if implementation:
                    route_implementations[mode].add(implementation)
    for mode, implementations in route_implementations.items():
        if len(implementations) > 1:
            messages.append(
                f"{mode}: routing implementations differ: {sorted(implementations)}"
            )
    return messages


def load_result_documents(paths: list[Path]) -> list[dict[str, Any]]:
    documents = [json.loads(path.read_text(encoding="utf-8")) for path in paths]
    messages = comparison_warnings(documents)
    if messages:
        print("Comparison warnings:")
        for message in messages:
            print(f"  WARNING: {message}")
    return documents


def print_status_table(documents: list[dict[str, Any]]) -> None:
    header = f"{'label':10s} {'formula':9s} {'AOs':>5s} {'mode':10s} {'runtime':12s} {'memory':12s}"
    print(header)
    print("-" * len(header))
    for document in documents:
        for molecule in document["molecules"].values():
            observed = molecule.get("observed") or {}
            aos = observed.get("actual_aos", molecule["expected_aos"])
            for mode in MODES:
                mode_record = molecule["modes"][mode]
                runtime_status = mode_record.get("runtime", {}).get("status", "pending")
                memory_status = mode_record.get("memory", {}).get("status", "pending")
                print(
                    f"{document['run_label'][:10]:10s} {molecule['formula']:9s} {aos:5d} "
                    f"{mode:10s} {runtime_status:12s} {memory_status:12s}"
                )


SELECTED_DOCUMENTS = (
    load_result_documents(SELECTED_RESULT_FILES) if SELECTED_RESULT_FILES else []
)
if SELECTED_DOCUMENTS:
    print_status_table(SELECTED_DOCUMENTS)
else:
    print("No benchmark result files were found. Run the benchmark script first.")

## Visualize

Each metric is rendered in its own notebook output with a single y-axis. Runtime curves show the median of the recorded samples with error bars spanning the observed minimum and maximum; one-sample legacy results therefore have zero-width bounds. Successful observations are plotted against actual spherical AO counts. Failed or timed-out points stay absent from curves and remain visible in the status table.

In [ ]:
from matplotlib.axes import Axes

MODE_COLORS = {
    "cpu": "#006D77",
    "cpu_dense": "#83C5BE",
    "gpu": "#C44536",
}
REVISION_LINESTYLES = ("-", "--", ":", "-.")
RESULT_MARKERS = ("o", "s", "^", "D", "v", "P", "X")
MARKER_SIZE = 5
LEGEND_MARKER_SCALE = 1.8
LEGEND_HANDLE_LENGTH = 3.0


def measurement_samples(mode_record: dict[str, Any], measurement: str) -> list[float]:
    result = mode_record.get(measurement, {})
    if result.get("status") != "ok":
        return []
    if measurement == "runtime":
        return [float(value) for value in result.get("runtime_samples_seconds", [])]
    peak_bytes = result.get("incremental_peak_bytes")
    return [float(peak_bytes) / 1024**3] if peak_bytes is not None else []


def sample_summary(samples: list[float]) -> tuple[float, float, float] | None:
    if not samples:
        return None
    values = np.asarray(samples, dtype=float)
    center = float(np.median(values))
    return center, center - float(values.min()), float(values.max()) - center


def measurement_value(mode_record: dict[str, Any], measurement: str) -> float | None:
    summary = sample_summary(measurement_samples(mode_record, measurement))
    return summary[0] if summary is not None else None


def measurement_series(
    document: dict[str, Any], mode: str, measurement: str
) -> tuple[list[int], list[float], list[float], list[float]]:
    points: list[tuple[int, float, float, float]] = []
    for molecule in document["molecules"].values():
        observed = molecule.get("observed") or {}
        aos = int(observed.get("actual_aos", molecule["expected_aos"]))
        summary = sample_summary(
            measurement_samples(molecule["modes"][mode], measurement)
        )
        if summary is not None and summary[0] > 0.0:
            points.append((aos, *summary))
    points.sort()
    return (
        [point[0] for point in points],
        [point[1] for point in points],
        [point[2] for point in points],
        [point[3] for point in points],
    )


def cpu_reference_ratio_series(
    document: dict[str, Any], measurement: str
) -> tuple[list[int], list[float], list[float], list[float]]:
    points: list[tuple[int, float, float, float]] = []
    for molecule in document["molecules"].values():
        observed = molecule.get("observed") or {}
        aos = int(observed.get("actual_aos", molecule["expected_aos"]))
        production_samples = measurement_samples(molecule["modes"]["cpu"], measurement)
        dense_samples = measurement_samples(molecule["modes"]["cpu_dense"], measurement)
        production_summary = sample_summary(production_samples)
        dense_summary = sample_summary(dense_samples)
        if (
            production_summary is None
            or dense_summary is None
            or min(production_samples) <= 0.0
        ):
            continue
        center = dense_summary[0] / production_summary[0]
        lower_bound = min(dense_samples) / max(production_samples)
        upper_bound = max(dense_samples) / min(production_samples)
        points.append((aos, center, center - lower_bound, upper_bound - center))
    points.sort()
    return (
        [point[0] for point in points],
        [point[1] for point in points],
        [point[2] for point in points],
        [point[3] for point in points],
    )


def endpoint_label(label: str, x_values: list[int]) -> str:
    return (
        f"{label} (last: {x_values[-1]} AOs)"
        if x_values
        else f"{label} (no successful points)"
    )


def style_benchmark_axis(
    axis: Axes, *, title: str, ylabel: str, logarithmic: bool = False
) -> None:
    axis.set(title=title, xlabel="Spherical AO count", ylabel=ylabel)
    if logarithmic:
        axis.set_yscale("log")
    axis.grid(True, which="both", color="#D9D9D9", linewidth=0.6)
    axis.legend(
        fontsize=8,
        loc="upper left",
        bbox_to_anchor=(1.02, 1.0),
        borderaxespad=0.0,
        markerscale=LEGEND_MARKER_SCALE,
        handlelength=LEGEND_HANDLE_LENGTH,
    )


def plot_measurement(documents: list[dict[str, Any]], measurement: str) -> None:
    if not documents:
        print(f"No result files selected; the {measurement} plot was not created.")
        return
    _, axis = plt.subplots(figsize=(11, 6), constrained_layout=True)
    for document_index, document in enumerate(documents):
        label = document["run_label"]
        line_style = REVISION_LINESTYLES[document_index % len(REVISION_LINESTYLES)]
        marker = RESULT_MARKERS[document_index % len(RESULT_MARKERS)]
        for mode in MODES:
            x_values, y_values, lower_errors, upper_errors = measurement_series(
                document, mode, measurement
            )
            curve_label = f"{label} {mode}"
            plot_arguments = {
                "color": MODE_COLORS[mode],
                "linestyle": line_style,
                "marker": marker,
                "markersize": MARKER_SIZE,
                "label": endpoint_label(curve_label, x_values),
            }
            if measurement == "runtime":
                axis.errorbar(
                    x_values,
                    y_values,
                    yerr=np.asarray([lower_errors, upper_errors]),
                    capsize=3,
                    **plot_arguments,
                )
            else:
                axis.plot(x_values, y_values, **plot_arguments)
    if measurement == "runtime":
        title = "One XC/Vxc evaluation (median and min-max)"
        ylabel = "Runtime (s)"
    elif measurement == "memory":
        title = "Incremental allocation peak"
        ylabel = "Memory (GiB)"
    else:
        raise ValueError(f"Unknown measurement: {measurement}")
    style_benchmark_axis(axis, title=title, ylabel=ylabel, logarithmic=True)
    plt.show()


def plot_cpu_reference_ratio(documents: list[dict[str, Any]], measurement: str) -> None:
    if not documents:
        print(
            f"No result files selected; the {measurement} ratio plot was not created."
        )
        return
    _, axis = plt.subplots(figsize=(11, 6), constrained_layout=True)
    for document_index, document in enumerate(documents):
        x_values, y_values, lower_errors, upper_errors = cpu_reference_ratio_series(
            document, measurement
        )
        line_style = REVISION_LINESTYLES[document_index % len(REVISION_LINESTYLES)]
        marker = RESULT_MARKERS[document_index % len(RESULT_MARKERS)]
        plot_arguments = {
            "color": MODE_COLORS["cpu"],
            "linestyle": line_style,
            "marker": marker,
            "markersize": MARKER_SIZE,
            "label": f"{document['run_label']} cpu",
        }
        if measurement == "runtime":
            axis.errorbar(
                x_values,
                y_values,
                yerr=np.asarray([lower_errors, upper_errors]),
                capsize=3,
                **plot_arguments,
            )
        else:
            axis.plot(x_values, y_values, **plot_arguments)
    if measurement == "runtime":
        title = "CPU production speedup (median and min-max)"
        ylabel = "CPU dense runtime / production runtime"
    elif measurement == "memory":
        title = "CPU production memory reduction"
        ylabel = "CPU dense peak / production peak"
    else:
        raise ValueError(f"Unknown measurement: {measurement}")
    style_benchmark_axis(axis, title=title, ylabel=ylabel)
    axis.axhline(1.0, color="#777777", linewidth=0.8, linestyle=":")
    plt.show()

In [ ]:
plot_measurement(SELECTED_DOCUMENTS, "runtime")

In [ ]:
plot_measurement(SELECTED_DOCUMENTS, "memory")

In [ ]:
plot_cpu_reference_ratio(SELECTED_DOCUMENTS, "runtime")

In [ ]:
plot_cpu_reference_ratio(SELECTED_DOCUMENTS, "memory")

## Numerical Differences

Production CPU and GPU fingerprints are compared molecule-by-molecule with the CPU-dense reference. The summary reports maximum absolute and relative errors and counts values outside the existing acceptance tolerances. Each per-fingerprint plot shows the signed difference `production - cpu_dense`; the dotted zero line is the CPU-dense reference.

The final six-panel figure compares CPU-dense references across result files. The first selected result is the baseline, and each curve shows `comparison cpu_dense - baseline cpu_dense`.

In [ ]:
FINGERPRINT_LABELS = {
    "electron_integral": "Electron integral",
    "xc_energy": "XC energy",
    "vxc_sum": "Vxc sum",
    "vxc_trace": "Vxc trace",
    "vxc_frobenius_norm": "Vxc Frobenius norm",
    "vxc_max_abs": "Vxc max abs",
}
ERROR_TOLERANCES = {
    "cpu": (1e-8, 5e-8),
    "gpu": (1e-7, 2e-7),
}


def fingerprint_error_records(
    document: dict[str, Any], production_mode: str, fingerprint_key: str
) -> list[dict[str, Any]]:
    rtol, atol = ERROR_TOLERANCES[production_mode]
    records: list[dict[str, Any]] = []
    for formula, molecule in document["molecules"].items():
        reference = preferred_fingerprint(molecule["modes"]["cpu_dense"])
        production = preferred_fingerprint(molecule["modes"][production_mode])
        if (
            reference is None
            or production is None
            or fingerprint_key not in reference
            or fingerprint_key not in production
        ):
            continue
        reference_value = float(reference[fingerprint_key])
        production_value = float(production[fingerprint_key])
        difference = production_value - reference_value
        absolute_error = abs(difference)
        relative_error = absolute_error / max(
            abs(reference_value), np.finfo(float).tiny
        )
        tolerance_scale = atol + rtol * abs(reference_value)
        observed = molecule.get("observed") or {}
        records.append(
            {
                "formula": formula,
                "aos": int(observed.get("actual_aos", molecule["expected_aos"])),
                "difference": difference,
                "absolute_error": absolute_error,
                "relative_error": relative_error,
                "tolerance_ratio": absolute_error / tolerance_scale,
            }
        )
    records.sort(key=lambda record: record["aos"])
    return records


def dense_reference_difference_records(
    reference_document: dict[str, Any],
    comparison_document: dict[str, Any],
    fingerprint_key: str,
) -> list[dict[str, Any]]:
    records: list[dict[str, Any]] = []
    for formula, reference_molecule in reference_document["molecules"].items():
        comparison_molecule = comparison_document["molecules"].get(formula)
        if comparison_molecule is None:
            continue
        reference = preferred_fingerprint(reference_molecule["modes"]["cpu_dense"])
        comparison = preferred_fingerprint(comparison_molecule["modes"]["cpu_dense"])
        if (
            reference is None
            or comparison is None
            or fingerprint_key not in reference
            or fingerprint_key not in comparison
        ):
            continue
        observed = comparison_molecule.get("observed") or {}
        records.append(
            {
                "formula": formula,
                "aos": int(
                    observed.get("actual_aos", comparison_molecule["expected_aos"])
                ),
                "difference": float(comparison[fingerprint_key])
                - float(reference[fingerprint_key]),
            }
        )
    records.sort(key=lambda record: record["aos"])
    return records


def print_fingerprint_error_summary(documents: list[dict[str, Any]]) -> None:
    header = (
        f"{'label':10s} {'mode':4s} {'fingerprint':22s} "
        f"{'max abs':>11s} {'max rel':>11s} {'outside':>8s} {'at':>8s}"
    )
    print(header)
    print("-" * len(header))
    for document in documents:
        for production_mode in ERROR_TOLERANCES:
            for fingerprint_key, fingerprint_label in FINGERPRINT_LABELS.items():
                records = fingerprint_error_records(
                    document, production_mode, fingerprint_key
                )
                if not records:
                    continue
                max_absolute_error = max(record["absolute_error"] for record in records)
                worst_relative = max(
                    records, key=lambda record: record["relative_error"]
                )
                outside_tolerance = sum(
                    record["tolerance_ratio"] > 1.0 for record in records
                )
                print(
                    f"{document['run_label'][:10]:10s} {production_mode:4s} "
                    f"{fingerprint_label:22s} {max_absolute_error:11.3e} "
                    f"{worst_relative['relative_error']:11.3e} "
                    f"{outside_tolerance:3d}/{len(records):<4d} "
                    f"{worst_relative['formula']:>8s}"
                )


def plot_fingerprint_differences(
    documents: list[dict[str, Any]], fingerprint_key: str
) -> None:
    if fingerprint_key not in FINGERPRINT_LABELS:
        raise ValueError(f"Unknown fingerprint: {fingerprint_key}")
    if not documents:
        print(f"No result files selected; the {fingerprint_key} plot was not created.")
        return
    _, axis = plt.subplots(figsize=(11, 6), constrained_layout=True)
    for document_index, document in enumerate(documents):
        line_style = REVISION_LINESTYLES[document_index % len(REVISION_LINESTYLES)]
        marker = RESULT_MARKERS[document_index % len(RESULT_MARKERS)]
        for production_mode in ERROR_TOLERANCES:
            records = fingerprint_error_records(
                document, production_mode, fingerprint_key
            )
            x_values = [record["aos"] for record in records]
            curve_label = f"{document['run_label']} {production_mode}"
            axis.plot(
                x_values,
                [record["difference"] for record in records],
                color=MODE_COLORS[production_mode],
                linestyle=line_style,
                marker=marker,
                markersize=MARKER_SIZE,
                label=endpoint_label(curve_label, x_values),
            )
    fingerprint_label = FINGERPRINT_LABELS[fingerprint_key]
    axis.axhline(
        0.0,
        color=MODE_COLORS["cpu_dense"],
        linewidth=1.0,
        linestyle=":",
        label="CPU dense reference",
    )
    style_benchmark_axis(
        axis,
        title=f"{fingerprint_label} difference from CPU dense",
        ylabel=f"{fingerprint_label} - CPU dense reference",
    )
    axis.ticklabel_format(axis="y", style="sci", scilimits=(0, 0))
    plt.show()


def plot_dense_reference_differences(documents: list[dict[str, Any]]) -> None:
    if len(documents) < 2:
        print("At least two result files are required to compare CPU-dense references.")
        return
    reference_document = documents[0]
    figure, axes = plt.subplots(2, 3, figsize=(16, 9), constrained_layout=True)
    for axis, (fingerprint_key, fingerprint_label) in zip(
        axes.flat, FINGERPRINT_LABELS.items(), strict=True
    ):
        plotted_differences: list[float] = []
        for document_index, document in enumerate(documents[1:], start=1):
            records = dense_reference_difference_records(
                reference_document, document, fingerprint_key
            )
            x_values = [record["aos"] for record in records]
            differences = [record["difference"] for record in records]
            plotted_differences.extend(differences)
            curve_label = f"{document['run_label']} - {reference_document['run_label']}"
            axis.plot(
                x_values,
                differences,
                color=MODE_COLORS["cpu_dense"],
                linestyle=REVISION_LINESTYLES[
                    document_index % len(REVISION_LINESTYLES)
                ],
                marker=RESULT_MARKERS[document_index % len(RESULT_MARKERS)],
                markersize=MARKER_SIZE,
                label=endpoint_label(curve_label, x_values),
            )
        axis.axhline(0.0, color="#777777", linewidth=0.8, linestyle=":")
        axis.set(
            title=fingerprint_label,
            xlabel="Spherical AO count",
            ylabel="CPU-dense difference",
        )
        if plotted_differences and all(value == 0.0 for value in plotted_differences):
            axis.set_ylim(-0.5, 0.5)
            axis.set_yticks([0.0])
            axis.text(
                0.5,
                0.54,
                "All matched differences are exactly zero",
                color="#555555",
                fontsize=8,
                ha="center",
                transform=axis.transAxes,
            )
        else:
            axis.ticklabel_format(axis="y", style="sci", scilimits=(0, 0))
        axis.grid(True, which="both", color="#D9D9D9", linewidth=0.6)
    handles, labels = axes.flat[0].get_legend_handles_labels()
    figure.legend(
        handles,
        labels,
        fontsize=8,
        loc="center left",
        bbox_to_anchor=(1.01, 0.5),
        borderaxespad=0.0,
        markerscale=LEGEND_MARKER_SCALE,
        handlelength=LEGEND_HANDLE_LENGTH,
    )
    figure.suptitle(
        f"CPU-dense reference differences from {reference_document['run_label']}"
    )
    plt.show()

In [ ]:
print_fingerprint_error_summary(SELECTED_DOCUMENTS)

In [ ]:
for fingerprint_key in FINGERPRINT_LABELS:
    plot_fingerprint_differences(SELECTED_DOCUMENTS, fingerprint_key)

In [ ]:
plot_dense_reference_differences(SELECTED_DOCUMENTS)